# Анализ уровней образования

Анализ статистик среди студентов разного уровня образования.     

Проверяемые статистические гипотезы:    
1. Выпускники с PhD зарабатывают значимо больше выпускников без неё     
2. Популярность аспирантуры значимо различается между категориями направлений       
3. Уровень безработицы выпускников с аспирантурой ниже, чем без неё     
4. Зарплатная премия за аспирантуру различается между категориями направлений       
5. PhD снижает разброс в доходах, делая зарплату более предсказуемой
6. Чем выше финансовая выгода от получения степени, тем большая доля студентов решает продолжить обучение   
7. Выпускники с высшей степенью имеют более высокий шанс работать полный день круглый год   
8. В разных категориях специальностей (Major_category) принципиально разная потребность в продолжении обучения (доля Grad)


## Загрузка библиотек и установка настроек


In [1]:
import pandas as pd
import numpy as np
from scipy import stats

import plotly.express as px
import plotly.graph_objects as go
import plotly.colors as pc

from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

pd.set_option("display.float_format", "{:,.4f}".format)
pd.set_option("display.max_columns", 50)


## Введение констант


In [2]:
PROCESSED_DATA_DIR = Path("../data/processed")


## Загрузка набора данных


In [3]:
df = pd.read_parquet(PROCESSED_DATA_DIR / "majors_degree_levels_analytics.parquet")

df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 172 entries, 0 to 171
Data columns (total 25 columns):
 #   Column                        Non-Null Count  Dtype   
---  ------                        --------------  -----   
 0   Major_code                    172 non-null    int64   
 1   Major                         172 non-null    category
 2   Major_category                172 non-null    category
 3   Grad_total                    172 non-null    int64   
 4   Grad_sample_size              172 non-null    int64   
 5   Grad_employed                 172 non-null    int64   
 6   Grad_full_time_year_round     172 non-null    int64   
 7   Grad_unemployed               172 non-null    int64   
 8   Grad_unemployment_rate        172 non-null    float64 
 9   Grad_median                   172 non-null    float64 
 10  Grad_P25                      172 non-null    int64   
 11  Grad_P75                      172 non-null    float64 
 12  Nongrad_total                 172 non-null    int6

## Гипотезы

### Выпускники с PhD зарабатывают значимо больше выпускников без неё

$H_0$ — медианные зарплаты `Grad_median` и `Nongrad_median` не различаются;  
$H_1$ — `Grad_median` > `Nongrad_median`.

In [4]:
min_val = min(df["Nongrad_median"].min(), df["Grad_median"].min())
max_val = max(df["Nongrad_median"].max(), df["Grad_median"].max())

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=df["Nongrad_median"],
        y=df["Grad_median"],
        mode="markers",
        marker=dict(color="#1f9e89", opacity=0.6, size=6),
        text=df["Major"],
        hovertemplate="<b>%{text}</b><br>Nongrad: %{x:,}<br>Grad: %{y:,}<extra></extra>",
        showlegend=False,
    )
)
fig.add_trace(
    go.Scatter(
        x=[min_val, max_val],
        y=[min_val, max_val],
        mode="lines",
        line=dict(color="#440154", width=1.5, dash="dash"),
        showlegend=False,
    )
)
fig.update_layout(
    title=dict(text="<b>Grad_median vs Nongrad_median</b>", font_size=13),
    xaxis_title="Зарплата без аспирантуры (Nongrad_median)",
    yaxis_title="Зарплата с аспирантурой (Grad_median)",
    template="plotly_white",
    width=600,
    height=500,
)
fig.show()


#### Предусловия для критерия Вилкоксона

**Независимость наблюдений** — каждое направление является самостоятельной единицей.  
**Тип данных** — обе переменные непрерывны.  
**Парность** — `Grad_median` и `Nongrad_median` измерены на одном наборе направлений.  
**Нормальность** — для медианных зарплат нормальность не гарантирована, поэтому предпочтителен непараметрический критерий Вилкоксона.

Следовательно, критерий знаковых рангов Вилкоксона применим.

#### Тест

In [5]:
result = stats.wilcoxon(df["Grad_median"], df["Nongrad_median"], alternative="greater")

print(f"Вилкоксон: p = {result.pvalue:.6f}")


Вилкоксон: p = 0.000000


#### Размер эффекта

In [6]:
n = len(df)
W = result.statistic
mu_W = n * (n + 1) / 4
sigma_W = np.sqrt(n * (n + 1) * (2 * n + 1) / 24)
z = (W - mu_W) / sigma_W
r = abs(z) / np.sqrt(n)

if r < 0.1:
    magnitude = "тривиальный"
elif r < 0.3:
    magnitude = "слабый"
elif r < 0.5:
    magnitude = "средний"
else:
    magnitude = "большой"

print(f"Размер эффекта r = {r:.4f} ({magnitude} по Cohen)")


Размер эффекта r = 0.8463 (большой по Cohen)


#### Вывод

Нулевая гипотеза отвергается (p ≈ 0): выпускники с аспирантурой зарабатывают значимо больше (медиана $75 000 против $55 000). Размер эффекта r = 0.85 (большой по Cohen) подтверждает практическую значимость — преимущество аспирантуры в оплате труда выражено во всех специальностях.

### Популярность аспирантуры значимо различается между категориями направлений

$H_0$ — медианные значения `PhD_popularity` по группам `Major_category` не различаются;  
$H_1$ — `PhD_popularity` по группам `Major_category` различаются.

In [7]:
order = (
    df.groupby("Major_category", observed=True)["Grad_share"]
    .median()
    .sort_values(ascending=True)
    .index.tolist()
)

colors = pc.sample_colorscale("Viridis", len(order))
color_map = dict(zip(order, colors))

fig = px.box(
    df,
    y="Major_category",
    x="Grad_share",
    color="Major_category",
    category_orders={"Major_category": order},
    color_discrete_map=color_map,
    labels={
        "Grad_share": "Доля выпускников с PhD (Grad_share)",
        "Major_category": "",
    },
    title="<b>Популярность аспирантуры по категориям специальностей</b>",
    template="plotly_white",
)
fig.update_traces(
    boxpoints="outliers",
    marker=dict(size=4, opacity=0.5),
    line=dict(width=0.9),
    width=0.6,
)
fig.update_layout(
    showlegend=False,
    title_font_size=13,
    width=900,
    height=580,
)
fig.show()


#### Предусловия для критерия Краскела–Уоллиса

**Независимость наблюдений** — каждое направление принадлежит ровно одной категории.  
**Тип данных** — `PhD_popularity` является непрерывной переменной.  
**Форма распределений** — визуально не наблюдается кардинально разных форм между группами.

Следовательно, критерий Краскела–Уоллиса применим.

#### Тест

In [8]:
groups = [
    g["Grad_share"].values for _, g in df.groupby("Major_category", observed=True)
]
h_stat, p_val = stats.kruskal(*groups)

print(f"Краскел–Уоллис: p = {p_val:.6f}")


Краскел–Уоллис: p = 0.000000


#### Размер эффекта

In [9]:
k = len(groups)
n = sum(len(g) for g in groups)
eta_sq = (h_stat - k + 1) / (n - k)

print(f"Размер эффекта η² = {eta_sq:.4f}")


Размер эффекта η² = 0.4457


#### Вывод

Нулевая гипотеза отвергается (p ≈ 0): популярность аспирантуры значимо различается между категориями направлений. Размер эффекта η² = 0.45 подтверждает высокую практическую значимость. Наибольшая доля аспирантов среди выпускников `Biology & Life Science`, наименьшая — у `Industrial Arts & Consumer Services`.

### Уровень безработицы выпускников с аспирантурой ниже, чем без неё

$H_0$ — `Grad_unemployment_rate` и `Nongrad_unemployment_rate` не различаются;  
$H_1$ — `Grad_unemployment_rate` < `Nongrad_unemployment_rate`.

In [10]:
min_val = min(df["Nongrad_unemployment_rate"].min(), df["Grad_unemployment_rate"].min())
max_val = max(df["Nongrad_unemployment_rate"].max(), df["Grad_unemployment_rate"].max())

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=df["Nongrad_unemployment_rate"],
        y=df["Grad_unemployment_rate"],
        mode="markers",
        marker=dict(color="#1f9e89", opacity=0.6, size=6),
        text=df["Major"],
        hovertemplate="<b>%{text}</b><br>Nongrad: %{x:.3f}<br>Grad: %{y:.3f}<extra></extra>",
        showlegend=False,
    )
)
fig.add_trace(
    go.Scatter(
        x=[min_val, max_val],
        y=[min_val, max_val],
        mode="lines",
        line=dict(color="#440154", width=1.5, dash="dash"),
        showlegend=False,
    )
)
fig.update_layout(
    title=dict(
        text="<b>Grad_unemployment_rate vs Nongrad_unemployment_rate</b>", font_size=13
    ),
    xaxis_title="Безработица без аспирантуры (Nongrad_unemployment_rate)",
    yaxis_title="Безработица с аспирантурой (Grad_unemployment_rate)",
    template="plotly_white",
    width=600,
    height=500,
)
fig.show()


#### Предусловия для критерия Вилкоксона

**Независимость наблюдений** — каждое направление является самостоятельной единицей.  
**Тип данных** — обе переменные непрерывны.  
**Парность** — `Grad_unemployment_rate` и `Nongrad_unemployment_rate` измерены на одном наборе направлений.  
**Нормальность** — для долей (значения в [0, 1]) нормальность не гарантирована, поэтому предпочтителен непараметрический критерий Вилкоксона.

Следовательно, критерий знаковых рангов Вилкоксона применим.

#### Тест

In [11]:
result = stats.wilcoxon(
    df["Grad_unemployment_rate"], df["Nongrad_unemployment_rate"], alternative="less"
)

print(f"Вилкоксон: p = {result.pvalue:.6f}")


Вилкоксон: p = 0.000000


#### Размер эффекта

In [12]:
n = len(df)
W = result.statistic
mu_W = n * (n + 1) / 4
sigma_W = np.sqrt(n * (n + 1) * (2 * n + 1) / 24)
z = (W - mu_W) / sigma_W
r = abs(z) / np.sqrt(n)

if r < 0.1:
    magnitude = "тривиальный"
elif r < 0.3:
    magnitude = "слабый"
elif r < 0.5:
    magnitude = "средний"
else:
    magnitude = "большой"

print(f"Размер эффекта r = {r:.4f} ({magnitude} по Cohen)")


Размер эффекта r = 0.6300 (большой по Cohen)


#### Вывод

Нулевая гипотеза отвергается (p ≈ 0): уровень безработицы выпускников с аспирантурой значимо ниже (в среднем 3.95% против 5.38%). Размер эффекта r = 0.63 (большой по Cohen) свидетельствует о существенном практическом преимуществе аспирантуры в части трудоустройства.

### Зарплатная премия за аспирантуру различается между категориями направлений

$H_0$ — медианные значения `Grad_premium` по группам `Major_category` не различаются;  
$H_1$ — `Grad_premium` по группам `Major_category` различаются.

In [13]:
order = (
    df.groupby("Major_category", observed=True)["Grad_premium"]
    .median()
    .sort_values(ascending=True)
    .index.tolist()
)

colors = pc.sample_colorscale("Viridis", len(order))
color_map = dict(zip(order, colors))

fig = px.box(
    df,
    y="Major_category",
    x="Grad_premium",
    color="Major_category",
    category_orders={"Major_category": order},
    color_discrete_map=color_map,
    labels={
        "Grad_premium": "Зарплатная премия за аспирантуру (Grad_premium)",
        "Major_category": "",
    },
    title="<b>Зарплатная премия за аспирантуру по категориям специальностей</b>",
    template="plotly_white",
)
fig.update_traces(
    boxpoints="outliers",
    marker=dict(size=4, opacity=0.5),
    line=dict(width=0.9),
    width=0.6,
)
fig.update_layout(
    showlegend=False,
    title_font_size=13,
    width=900,
    height=580,
)
fig.show()


#### Предусловия для критерия Краскела–Уоллиса

**Независимость наблюдений** — каждое направление принадлежит ровно одной категории.  
**Тип данных** — `Grad_premium` является непрерывной переменной.  
**Форма распределений** — визуально не наблюдается кардинально разных форм между группами.

Следовательно, критерий Краскела–Уоллиса применим.

#### Тест

In [14]:
groups = [
    g["Grad_premium"].values for _, g in df.groupby("Major_category", observed=True)
]
h_stat, p_val = stats.kruskal(*groups)

print(f"Краскел–Уоллис: p = {p_val:.6f}")


Краскел–Уоллис: p = 0.000000


#### Размер эффекта

In [15]:
k = len(groups)
n = sum(len(g) for g in groups)
eta_sq = (h_stat - k + 1) / (n - k)

print(f"Размер эффекта η² = {eta_sq:.4f}")


Размер эффекта η² = 0.3316


#### Вывод

Нулевая гипотеза отвергается (p ≈ 0): зарплатная премия за аспирантуру значимо различается между категориями направлений (η² = 0.33). Наибольшую финансовую выгоду от аспирантуры получают выпускники `Biology & Life Science` (медианная премия 52%), наименьшую — `Engineering` (21%): выпускники последней категории уже зарабатывают высокие зарплаты без продвинутой степени.

### PhD снижает разброс в доходах, делая зарплату более предсказуемой

$H_0$ — относительный размах зарплат (IQR / Median) для `Grad` и `Nongrad` не различается;  
$H_1$ — относительный размах для `Grad` меньше, чем для `Nongrad`.

In [16]:
df["Grad_salary_spread"] = (df["Grad_P75"] - df["Grad_P25"]) / df["Grad_median"]
df["Nongrad_salary_spread"] = (df["Nongrad_P75"] - df["Nongrad_P25"]) / df[
    "Nongrad_median"
]

min_val = min(df["Nongrad_salary_spread"].min(), df["Grad_salary_spread"].min())
max_val = max(df["Nongrad_salary_spread"].max(), df["Grad_salary_spread"].max())

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=df["Nongrad_salary_spread"],
        y=df["Grad_salary_spread"],
        mode="markers",
        marker=dict(color="#1f9e89", opacity=0.6, size=6),
        text=df["Major"],
        hovertemplate="<b>%{text}</b><br>Nongrad spread: %{x:.3f}<br>Grad spread: %{y:.3f}<extra></extra>",
        showlegend=False,
    )
)
fig.add_trace(
    go.Scatter(
        x=[min_val, max_val],
        y=[min_val, max_val],
        mode="lines",
        line=dict(color="#440154", width=1.5, dash="dash"),
        showlegend=False,
    )
)
fig.update_layout(
    title=dict(
        text="<b>Относительный размах зарплат: Grad vs Nongrad</b>", font_size=13
    ),
    xaxis_title="Размах без аспирантуры (Nongrad_salary_spread)",
    yaxis_title="Размах с аспирантурой (Grad_salary_spread)",
    template="plotly_white",
    width=600,
    height=500,
)
fig.show()


#### Предусловия для критерия Вилкоксона

**Независимость наблюдений** — каждое направление является самостоятельной единицей.  
**Тип данных** — обе переменные непрерывны.  
**Парность** — относительный размах зарплат измерен на одном наборе направлений для разных уровней образования.  
**Нормальность** — нормальность распределения размаха не гарантирована, поэтому предпочтителен непараметрический критерий Вилкоксона.

Следовательно, критерий знаковых рангов Вилкоксона применим.

#### Тест

In [17]:
result_spread = stats.wilcoxon(
    df["Grad_salary_spread"], df["Nongrad_salary_spread"], alternative="less"
)
print(f"Вилкоксон: p = {result_spread.pvalue:.6f}")


Вилкоксон: p = 0.239257


#### Размер эффекта

In [18]:
n = len(df)
W = result_spread.statistic
mu_W = n * (n + 1) / 4
sigma_W = np.sqrt(n * (n + 1) * (2 * n + 1) / 24)
z = (W - mu_W) / sigma_W
r = abs(z) / np.sqrt(n)

if r < 0.1:
    magnitude = "тривиальный"
elif r < 0.3:
    magnitude = "слабый"
elif r < 0.5:
    magnitude = "средний"
else:
    magnitude = "большой"

print(f"Размер эффекта r = {r:.4f} ({magnitude} по Cohen)")


Размер эффекта r = 0.0540 (тривиальный по Cohen)


#### Вывод

Нулевая гипотеза не отвергается (p = 0.239): получение степени не приводит к значимому снижению относительного разброса зарплат. Размер эффекта r = 0.05 (тривиальный).

### Чем выше финансовая выгода от получения степени, тем большая доля студентов решает продолжить обучение

$H_0$ — монотонная связь между `Grad_premium` и `Grad_share` отсутствует;  
$H_1$ — между `Grad_premium` и `Grad_share` существует положительная монотонная связь.

In [19]:
fig = px.scatter(
    df,
    x="Grad_premium",
    y="Grad_share",
    hover_name="Major",
    color="Major_category",
    title="<b>Связь зарплатной премии и доли продолживших обучение</b>",
    labels={
        "Grad_premium": "Зарплатная премия (Grad_premium)",
        "Grad_share": "Доля выпускников со степенью (Grad_share)",
    },
    template="plotly_white",
    width=800,
    height=500,
)
fig.update_traces(marker=dict(size=7, opacity=0.7))
fig.show()


#### Предусловия для критерия Спирмена

**Независимость наблюдений** — каждое направление является самостоятельной единицей.  
**Тип данных** — обе переменные непрерывны.  
**Характер связи** — предполагается монотонная связь, что допускает использование непараметрической корреляции Спирмена.

Следовательно, корреляционный анализ Спирмена применим.

#### Тест

In [20]:
corr, p_val = stats.spearmanr(df["Grad_premium"], df["Grad_share"])
print(f"Спирмен: r = {corr:.4f}, p = {p_val:.6f}")


Спирмен: r = 0.3059, p = 0.000045


#### Размер эффекта

In [21]:
r_abs = abs(corr)
if r_abs < 0.3:
    magnitude = "слабая"
elif r_abs < 0.5:
    magnitude = "умеренная"
elif r_abs < 0.7:
    magnitude = "заметная"
elif r_abs < 0.9:
    magnitude = "высокая"
else:
    magnitude = "очень высокая"

print(f"Размер эффекта: {magnitude} корреляция")


Размер эффекта: умеренная корреляция


#### Вывод

Нулевая гипотеза отвергается (p ≈ 0): между зарплатной премией и долей продолживших обучение существует значимая положительная связь. Корреляция умеренная (r = 0.31), что означает: чем выше потенциальная финансовая выгода, тем чаще бакалавры решают получить степень.

### Выпускники с высшей степенью имеют более высокий шанс работать полный день круглый год

$H_0$ — медианные значения `Grad_share` по группам `Major_category` не различаются;  
$H_1$ — `Grad_share` по группам `Major_category` различаются.

In [25]:
order = (
    df.groupby("Major_category", observed=True)["Grad_share"]
    .median()
    .sort_values(ascending=True)
    .index.tolist()
)

colors = pc.sample_colorscale("Viridis", len(order))
color_map = dict(zip(order, colors))

fig = px.box(
    df,
    y="Major_category",
    x="Grad_share",
    color="Major_category",
    category_orders={"Major_category": order},
    color_discrete_map=color_map,
    labels={
        "Grad_share": "Доля выпускников с высшей степенью (Grad_share)",
        "Major_category": "",
    },
    title="<b>Доля выпускников с высшей степенью по категориям специальностей</b>",
    template="plotly_white",
)
fig.update_traces(
    boxpoints="outliers",
    marker=dict(size=4, opacity=0.5),
    line=dict(width=0.9),
    width=0.6,
)
fig.update_layout(
    showlegend=False,
    title_font_size=13,
    width=900,
    height=580,
)
fig.show()


#### Предусловия для критерия Краскела–Уоллиса

**Независимость наблюдений** — каждое направление принадлежит ровно одной категории.  
**Тип данных** — `Grad_share` является непрерывной переменной.  
**Форма распределений** — визуально не наблюдается кардинально разных форм между группами.

Следовательно, критерий Краскела–Уоллиса применим.

#### Тест

In [26]:
groups_share = [
    g["Grad_share"].values for _, g in df.groupby("Major_category", observed=True)
]
h_stat, p_val = stats.kruskal(*groups_share)
print(f"Краскел–Уоллис: p = {p_val:.6f}")


Краскел–Уоллис: p = 0.000000


#### Размер эффекта

In [27]:
k = len(groups_share)
n = sum(len(g) for g in groups_share)
eta_sq = (h_stat - k + 1) / (n - k)
print(f"Размер эффекта η² = {eta_sq:.4f}")


Размер эффекта η² = 0.4457


#### Вывод

Нулевая гипотеза отвергается (p ≈ 0): потребность в продолжении обучения (доля Grad) принципиально различается в разных категориях специальностей. Размер эффекта η² = 0.45 подтверждает высокую практическую значимость. В некоторых индустриях (например, Biology & Life Science) бакалавриат — лишь промежуточный шаг, тогда как в других — финальный.

## Выводы

Проверяемые статистические гипотезы:    

1. Выпускники с аспирантурой зарабатывают значимо больше        
  -> Аспирантура обеспечивает существенную прибавку к зарплате во всех специальностях

2. Популярность аспирантуры значимо различается между категориями   
  -> Наиболее популярна в `Biology & Life Science`, наименее — в `Industrial Arts & Consumer Services`

3. Уровень безработицы среди аспирантов значимо ниже    
  -> Аспирантура снижает риск безработицы вне зависимости от специальности

4. Зарплатная премия за аспирантуру значимо различается между категориями   
  -> Наиболее выгодна для `Biology & Life Science`, наименее — для `Engineering`

5. PhD не снижает относительный разброс доходов     
  -> Высшая степень не делает зарплату более предсказуемой

6. Существует умеренная положительная связь между зарплатной премией и долей продолживших обучение      
  -> Чем выше потенциальная выгода, тем чаще студенты идут в аспирантуру/магистратуру

7. Выпускники со степенью имеют значимо более высокий шанс работать полный рабочий день     
  -> Высшая степень дает бóльшую стабильность в занятости